In [4]:
from dataclasses import dataclass
from pathlib import Path

In [44]:
@dataclass()
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    target_column: str
    date_column: str
    model_name: str
    order: tuple
 
    

In [ ]:
import os
# os.chdir("Stock-Price-Prediction-MLOps")
!pwd

/c/Users/Ibk/Desktop/data project/Stock-Price-Prediction-MLOps


In [63]:
from stock_prediction.utils.common import *
from stock_prediction.constants import *
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, schema_filepath=SCHEMA_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.schema = read_yaml(schema_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
        
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer

        create_directories([config.root_dir])
        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            model_name=config.model_name,
            date_column=self.schema.date_column,
            target_column=self.schema.target_column.name,
            order=self.params.ARIMA.order
           
        )
        return model_trainer_config
        

In [71]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from stock_prediction import logger


class ModelTrainer:
    def __init__(self, config):
      self.config = config
    def train(self) -> ARIMA:
      train_df = pd.read_csv(self.config.train_data_path, parse_dates=[self.config.date_column])
      
      train = train_df.set_index(self.config.date_column)[self.config.target_column]
  
      arima_model = ARIMA(train, order=self.config.order)
      logger.info("Model training complete")
      # arima_model.fit()
      model_path =  Path(self.config.root_dir) / self.config.model_name
      save_bin(arima_model, model_path)
      logger.info(f"Arima model saved to {model_path}")
      return arima_model
      
    
    

In [72]:
config = ConfigurationManager()
model_trainer_config = config.get_model_trainer_config()
model_trainer = ModelTrainer(model_trainer_config)
model_trainer.train()

2026-08-11 15:36:26,983 | INFO | common| YAML file: config\config.yaml loaded successfully.
2026-08-11 15:36:26,987 | INFO | common| YAML file: schema.yaml loaded successfully.
2026-08-11 15:36:26,989 | INFO | common| YAML file: params.yaml loaded successfully.
2026-08-11 15:36:26,991 | INFO | common| Directory created at: artifacts
2026-08-11 15:36:26,993 | INFO | common| Directory created at: artifacts/model_trainer
2026-08-11 15:36:27,007 | INFO | 96704062| Model training complete
2026-08-11 15:36:27,019 | INFO | common| Binary file saved at: artifacts\model_trainer\arima.joblib
2026-08-11 15:36:27,021 | INFO | 96704062| Arima model saved to artifacts\model_trainer\arima.joblib


c:\Users\Ibk\Desktop\data project\Stock-Price-Prediction-MLOps\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Ibk\Desktop\data project\Stock-Price-Prediction-MLOps\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Ibk\Desktop\data project\Stock-Price-Prediction-MLOps\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [ ]:
# src/stockProject/components/model_trainer.py
import pandas as pd
import joblib
import os
from pathlib import Path
from statsmodels.tsa.arima.model import ARIMA
from stockProject import logger
from stockProject.entity.config_entity import ModelTrainerConfig

class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        train_df = pd.read_csv(self.config.train_data_path, parse_dates=)

        train_series = train_df.set_index(self.config.date_column)[self.config.target_column]
        train_series.index = pd.DatetimeIndex(train_series.index).to_period(self.config.frequency)

        logger.info(f"Training ARIMA{tuple(self.config.order)} on {len(train_series)} observations")

        model = ARIMA(train_series, order=tuple(self.config.order))
        model_fit = model.fit()

        model_path = os.path.join(self.config.root_dir, self.config.model_name)
        joblib.dump(model_fit, model_path)
        logger.info(f"ARIMA model saved to: {model_path}")

        return model_fit